In [9]:
!pip install pandas scikit-learn catboost xgboost seaborn matplotlib joblib

In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import joblib, os, warnings, seaborn as sns, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
RANDOM_STATE = 42

In [13]:
df = pd.read_csv("cars_feature_engineered.csv")

# quick sanity checks
print(df.shape)
print(df.isna().sum().sort_values(ascending=False)[:10])
print(df.select_dtypes("object").nunique().sort_values(ascending=False)[:10])

# drop obvious leakage or unusable columns
drop_cols = ["vin", "saledate", "price_diff"]   # price_diff = target leakage
df = df.drop(columns=drop_cols)

# separate target
y = df["sellingprice"]
X = df.drop(columns=["sellingprice"])

(472325, 22)
year                0
make                0
sale_month          0
sale_year           0
mileage_per_year    0
car_age             0
price_diff          0
saledate            0
sellingprice        0
mmr                 0
dtype: int64
vin         465768
seller       11923
saledate      3609
trim          1494
model          768
make            53
state           34
color           20
interior        17
body             9
dtype: int64


In [15]:
#Identify column types
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print(f"{len(cat_cols)} categorical  : {cat_cols}")
print(f"{len(num_cols)} numerical    : {num_cols}")

9 categorical  : ['make', 'model', 'trim', 'body', 'transmission', 'state', 'color', 'interior', 'seller']
9 numerical    : ['year', 'condition', 'odometer', 'mmr', 'car_age', 'mileage_per_year', 'sale_year', 'sale_month', 'sale_day']


In [19]:
#Split for test and training before any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [23]:
#Helper for eval
def regression_report(model, X_tr, y_tr, X_te, y_te, name="model"):
    preds_tr = model.predict(X_tr)
    preds_te = model.predict(X_te)
    mae_tr  = mean_absolute_error(y_tr, preds_tr)
    mae_te  = mean_absolute_error(y_te, preds_te)
    rmse_tr = mean_squared_error(y_tr, preds_tr, squared=False)
    rmse_te = mean_squared_error(y_te, preds_te, squared=False)
    print(f"{name:15s} | MAE train {mae_tr:7.0f}  test {mae_te:7.0f} | RMSE train {rmse_tr:8.0f}  test {rmse_te:8.0f}")
    return {"mae_te": mae_te, "rmse_te": rmse_te}

In [29]:
# Assuming df is your DataFrame; adjust columns as needed

# Prepare X and y
X = df.drop('sellingprice', axis=1)
y = df['sellingprice']

# Preprocessor: One-hot encode cats, passthrough nums
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# Apply preprocessing
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# Now fit the model (X_train_encoded is all numerical)
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train_encoded, y_train)

# Your report function (adapt if it needs encoded data)
rf_scores = regression_report(rf, X_train_encoded, y_train, X_test_encoded, y_test, "RandomForest")

MemoryError: Unable to allocate 37.7 GiB for an array with shape (377860, 13394) and data type float64